# Assignment 3: Milestone I — Natural Language Processing
## Tasks 2 & 3: Feature Representations & Classification

#### Group: UN_Group 3
#### Student Name:
- Mai Dang Khoa ( s3974876 )
- Dang Cuu Dang Khoa ( s3979159 )
- Tran Quang Minh ( s3988776 )

Environment: Python 3 + Jupyter Notebook

Libraries used:
* pandas
* numpy
* gensim (`downloader`, `corpora.Dictionary`, `models.TfidfModel`)
* collections (`Counter`)

## Task 2 — Introduction

This notebook generates three feature representations from the cleaned `review_text`
produced by Task 1:

1. **Count vectors** — sparse bag-of-words against `vocab.txt`.
2. **Unweighted embedding vectors** — sum of FastText word vectors per review.
3. **TF-IDF weighted embedding vectors** — TF-IDF weighted sum of FastText vectors.

Chosen embedding: `fasttext-wiki-news-subwords-300` (300-dim) loaded via gensim
downloader. Note that this distribution ships only the trained word vectors
(`KeyedVectors`), not the full FastText `.bin` with subword n-gram parameters,
so OOV tokens are skipped (matching the lecturer's slide-49 try/except pattern).
We chose this model because its trained vocabulary (~1M tokens) is much larger
than Word2Vec GoogleNews-300 and includes informal/social-media words that
appear in beauty product reviews.

## Importing libraries 

In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter

import gensim.downloader as api
from gensim.corpora import Dictionary
from gensim.models import TfidfModel

OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Task 2. Generating Feature Representations for Cosmetics / Beauty Reviews

### 2.1 Load preprocessed data

Load:
- `processed.csv` — Task 1 output with the cleaned `review_text` column.
- `vocab.txt` — 0-indexed `word:index` dictionary built in Task 1.

The `review_text` column already holds space-separated cleaned tokens, so we
just need `.split()` to recover the token lists. We also keep `review_id` so
each output line can be tagged with `#<review_id>` per the spec.

The pipeline below builds three document representations end-to-end:

1. **Sparse Bag-of-Words** based on the Task-1 vocabulary (`vocab.txt`).
2. **Dense Bag of Embeddings** — sum of FastText word vectors per review.
3. **Dense TF-IDF Weighted Bag of Embeddings** — weighted sum using
   gensim's `TfidfModel`.

Each representation is generated, validated, and saved to disk in the
spec-required format `#<review_id>,...`.

In [2]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, "processed.csv"))
df["review_text"] = df["review_text"].fillna("")

# Recover token lists from the space-joined cleaned text saved by Task 1
tokenized = [text.split() for text in df["review_text"]]
review_ids = df["review_id"].tolist()

# Load vocab.txt → word2idx dict
word2idx = {}
with open(os.path.join(OUTPUT_DIR, "vocab.txt"), "r", encoding="utf-8") as f:
    for line in f:
        w, i = line.rstrip("\n").rsplit(":", 1)
        word2idx[w] = int(i)

print(f"Reviews loaded     : {len(df):,}")
print(f"Vocabulary size    : {len(word2idx):,}")
print(f"Empty reviews      : {sum(1 for t in tokenized if not t):,}")
print(f"Sample tokens (#0) : {tokenized[0][:10]}")

Reviews loaded     : 61,284
Vocabulary size    : 8,054
Empty reviews      : 1,911
Sample tokens (#0) : ['works', 'claims', 'difference', 'day', 'olay', 'cleanser', 'results']


### 2.2 Count Vectors (Bag-of-Words)
For each review we count the occurrences of each vocabulary word and write a
sparse line in the format required by the spec (page 3):

#<review_id>,word_idx:freq,word_idx:freq,...

Word-index entries are sorted **ascending**, matching the example in the
assignment PDF. Empty reviews produce a header-only line `#<review_id>,`.

In [3]:
out_path = os.path.join(OUTPUT_DIR, "count_vectors.txt")

with open(out_path, "w", encoding="utf-8") as fout:
    for rid, tokens in zip(review_ids, tokenized):
        counts = Counter(t for t in tokens if t in word2idx)
        items  = sorted(counts.items(), key=lambda kv: word2idx[kv[0]])
        body   = ",".join(f"{word2idx[w]}:{c}" for w, c in items)
        fout.write(f"#{rid},{body}\n")

print(f"Wrote {len(review_ids):,} lines to {out_path}")
print("\nSample (first 3 lines):")
with open(out_path) as f:
    for _ in range(3):
        print(f.readline().rstrip()[:140])

Wrote 61,284 lines to ../output/count_vectors.txt

Sample (first 3 lines):
#16752142,1140:1,1160:1,1706:1,1873:1,4828:1,5879:1,7940:1
#14682550,1140:1,4158:1,6509:1,6552:1,7182:1,7573:1
#15618995,15:1,1286:1,1972:1,3035:1,4474:1,4497:1,4818:1,5616:1,7882:1,7931:1


### 2.3 Load pretrained FastText embeddings

`fasttext-wiki-news-subwords-300` via gensim downloader (300-dim, ~1GB on first
download, cached after that). The gensim downloader provides the trained
`KeyedVectors` only, so OOV tokens are skipped (slide-49 try/except pattern)
rather than computed from subwords. We chose this model over GoogleNews-300
because its ~1M-token trained vocab includes more informal English, giving
better coverage for the typo/slang-heavy beauty-review corpus.

In [4]:
print("Loading FastText (first run downloads ~1GB) ...")
ft = api.load("fasttext-wiki-news-subwords-300")

DIM = ft.vector_size
print(f"FastText loaded — embedding dim: {DIM}")
print(f"Trained vocab size: {len(ft):,}")

in_trained = sum(1 for w in word2idx if w in ft.key_to_index)
print(f"\nTask-1 vocab in FastText trained vocab: "
      f"{in_trained:,} / {len(word2idx):,} ({in_trained/len(word2idx)*100:.1f}%)")
print(f"OOV tokens (the remaining {len(word2idx)-in_trained:,}) will be skipped "
      f"in the document-vector sum (slide 49 try/except pattern).")

Loading FastText (first run downloads ~1GB) ...
FastText loaded — embedding dim: 300
Trained vocab size: 999,999

Task-1 vocab in FastText trained vocab: 7,072 / 8,054 (87.8%)
OOV tokens (the remaining 982) will be skipped in the document-vector sum (slide 49 try/except pattern).


In [5]:
all_tokens = [t for review in tokenized for t in review]
oov_total  = sum(1 for t in all_tokens if t not in ft.key_to_index)

print(f"Total token occurrences across corpus : {len(all_tokens):,}")
print(f"OOV occurrences (will be skipped)     : {oov_total:,} "
      f"({oov_total/len(all_tokens)*100:.2f}%)")

oov_unique = sorted({t for t in all_tokens if t not in ft.key_to_index})
print(f"Unique OOV tokens : {len(oov_unique):,}")
print(f"\nExamples of OOV tokens (first 15): {oov_unique[:15]}")

Total token occurrences across corpus : 434,032
OOV occurrences (will be skipped)     : 8,306 (1.91%)
Unique OOV tokens : 982

Examples of OOV tokens (first 15): ['acene', 'acetone-free', 'acne-prone', 'afforadable', 'afordable', 'aftee', 'agarbatti', 'agarpathi', 'alil', 'aloevera', 'alottt', 'alovera', 'alsoo', 'amaazing', 'amaizing']


### 2.4 Unweighted embedding vectors

For each review we **sum** the FastText vectors of every token in the FastText
trained vocabulary (slide 49 style — try/except pattern, OOV tokens skipped).
Empty reviews and all-OOV reviews receive a zero vector. Each output line:

`#<review_id>,v1,v2,...,v300`

Values are written with 6 decimal places.


In [6]:
zero_vec = np.zeros(DIM, dtype=np.float32)

def doc_vec_sum(tokens):
    """Sum of FastText vectors, skipping OOV tokens (slide 49 style)."""
    vecs = [ft[w] for w in tokens if w in ft.key_to_index]
    if not vecs:
        return zero_vec
    return np.sum(np.stack(vecs), axis=0)

out_path = os.path.join(OUTPUT_DIR, "unweighted_vectors.txt")
with open(out_path, "w", encoding="utf-8") as fout:
    for rid, tokens in zip(review_ids, tokenized):
        v = doc_vec_sum(tokens)
        body = ",".join(f"{x:.6f}" for x in v)
        fout.write(f"#{rid},{body}\n")

print(f"Wrote {len(review_ids):,} lines to {out_path}")
print(f"Each line has {DIM} comma-separated values after the header.")
print("\nSample (first 80 chars of first 2 lines):")
with open(out_path) as f:
    for _ in range(2):
        print(f.readline()[:80] + " ...")

Wrote 61,284 lines to ../output/unweighted_vectors.txt
Each line has 300 comma-separated values after the header.

Sample (first 80 chars of first 2 lines):
#16752142,-0.060103,-0.168091,0.286550,0.036978,-0.016895,-0.204752,0.080907,-0. ...
#14682550,-0.024547,0.207373,0.017198,-0.078751,-0.036110,-0.175684,0.118396,-0. ...


### 2.5 TF-IDF weighted embedding vectors

Following the gensim approach on slide 51:

1. Build a `Dictionary` from the cleaned tokens.
2. Convert each review to bag-of-words with `doc2bow`.
3. Fit a `TfidfModel` on that corpus.
4. For each review:  `h = Σ tfidf(w, d) · ft[w]`.

Empty reviews receive a zero vector.

In [7]:
print("Building gensim Dictionary + TfidfModel ...")
docs_dict   = Dictionary(tokenized)
docs_corpus = [docs_dict.doc2bow(doc) for doc in tokenized]
tfidf_model = TfidfModel(docs_corpus, id2word=docs_dict)

print(f"Dictionary size : {len(docs_dict):,}")
print(f"Corpus length   : {len(docs_corpus):,}")
print("TfidfModel fitted.")

Building gensim Dictionary + TfidfModel ...
Dictionary size : 8,054
Corpus length   : 61,284
TfidfModel fitted.


In [8]:
def doc_vec_weighted(bow_doc):
    """TF-IDF weighted sum of FastText vectors, skipping OOV tokens."""
    if not bow_doc:
        return zero_vec
    v = np.zeros(DIM, dtype=np.float32)
    for word_id, weight in tfidf_model[bow_doc]:
        token = docs_dict[word_id]
        if token in ft.key_to_index:        # skip OOV
            v += weight * ft[token]
    return v

out_path = os.path.join(OUTPUT_DIR, "weighted_vectors.txt")
with open(out_path, "w", encoding="utf-8") as fout:
    for rid, bow in zip(review_ids, docs_corpus):
        v = doc_vec_weighted(bow)
        body = ",".join(f"{x:.6f}" for x in v)
        fout.write(f"#{rid},{body}\n")

print(f"Wrote {len(review_ids):,} lines to {out_path}")
print("\nSample (first 80 chars of first 2 lines):")
with open(out_path) as f:
    for _ in range(2):
        print(f.readline()[:80] + " ...")

Wrote 61,284 lines to ../output/weighted_vectors.txt

Sample (first 80 chars of first 2 lines):
#16752142,-0.023456,-0.043792,0.101615,0.022236,-0.013132,-0.068738,0.035675,-0. ...
#14682550,-0.015626,0.075160,0.001219,-0.021432,-0.015518,-0.058769,0.050817,-0. ...


### 2.6 Sanity checks

Verify line counts, vector dimensionality, and spot-check one review end-to-end.

In [9]:
def line_count(path):
    with open(path) as f:
        return sum(1 for _ in f)

paths = [os.path.join(OUTPUT_DIR, p) for p in
         ["count_vectors.txt", "unweighted_vectors.txt", "weighted_vectors.txt"]]
for p in paths:
    print(f"{os.path.basename(p):30s}  lines = {line_count(p):,}")

def dense_dims(path, n=10):
    dims = set()
    with open(path) as f:
        for i, line in enumerate(f):
            if i == n: break
            _, body = line.rstrip().split(",", 1)
            dims.add(len(body.split(",")))
    return dims

print(f"\nDense dim — unweighted (first 10 lines): "
      f"{dense_dims(os.path.join(OUTPUT_DIR, 'unweighted_vectors.txt'))}")
print(f"Dense dim — weighted   (first 10 lines): "
      f"{dense_dims(os.path.join(OUTPUT_DIR, 'weighted_vectors.txt'))}")

print("\n--- Spot check on row 0 ---")
print(f"review_id : {review_ids[0]}")
print(f"tokens    : {tokenized[0][:12]} ...")
with open(os.path.join(OUTPUT_DIR, "count_vectors.txt")) as f:
    print(f"count     : {f.readline().rstrip()[:140]} ...")

assert all(line_count(p) == len(df) for p in paths), "Line count mismatch!"
print("\nAll line counts == number of reviews.")

count_vectors.txt               lines = 61,284
unweighted_vectors.txt          lines = 61,284
weighted_vectors.txt            lines = 61,284

Dense dim — unweighted (first 10 lines): {300}
Dense dim — weighted   (first 10 lines): {300}

--- Spot check on row 0 ---
review_id : 16752142
tokens    : ['works', 'claims', 'difference', 'day', 'olay', 'cleanser', 'results'] ...
count     : #16752142,1140:1,1160:1,1706:1,1873:1,4828:1,5879:1,7940:1 ...

All line counts == number of reviews.


### 2.7 Summary

Three feature files written to `../output/` (configured via `OUTPUT_DIR`):

| File | Format | Source |
|------|--------|--------|
| `count_vectors.txt` | `#<review_id>,wIdx:freq,...` (sparse, sorted asc) | `vocab.txt` |
| `unweighted_vectors.txt` | `#<review_id>,v1,...,v300` (dense) | sum of FastText vectors |
| `weighted_vectors.txt` | `#<review_id>,v1,...,v300` (dense) | TF-IDF × FastText, summed |

These three files are the inputs for Task 3, where we'll train classifiers
that predict `is_a_buyer` and compare model performance across the three
representations using 5-fold cross-validation.

Before submission: export this notebook as `task2_3.py` via *File → Download as → Python (.py)*.

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [10]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>